# RI-JK RHF Hessian：CP-HF 分解 (3) Krylov

In [1]:
from pyscf import gto, scf, lib, df, hessian
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper
import scipy
from scipy.linalg import solve_triangular

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_cphf = np.load("nh3_r_hf_decomp.npz")["de_cphf"]

In [6]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)
occ_energy = mo_energy[mo_occ > 0]

In [7]:
eocc = mo_energy[mo_occ > 0]
evir = mo_energy[mo_occ == 0]
mvir = mo_coeff[:, mo_occ == 0]

## Overview

In [8]:
def ovlp_deriv1_generator(mol):
    int1e_ipovlp = mol.intor("int1e_ipovlp")
    
    def get_ovlp_deriv_at_atoms(A):
        shl0, shl1, p0, p1 = aoslices[A]
        s1ao = np.zeros((3, nao, nao))
        s1ao[:, p0:p1, :] += - int1e_ipovlp[:, p0:p1] 
        s1ao[:, :, p0:p1] += - int1e_ipovlp[:, p0:p1].transpose(0, 2, 1)
        return s1ao
    return get_ovlp_deriv_at_atoms

In [9]:
f1ao = mf_hess.make_h1(mo_coeff, mo_occ)
s1ao = np.array([ovlp_deriv1_generator(mol)(A) for A in range(natm)])
mo1, mo_e1 = mf_hess.solve_mo1(mo_energy, mo_coeff, mo_occ, f1ao)
mo1 = np.array(mo1)
mo_e1 = np.array(mo_e1)

## Problem-Setting

In [10]:
s1ao = np.array(s1ao).reshape(-1, nao, nao)
f1ao = np.array(f1ao).reshape(-1, nao, nao)
f1mo = mo_coeff.T @ f1ao @ mocc
s1mo = mo_coeff.T @ s1ao @ mocc
hsmo = f1mo - s1mo * eocc
e_ai = 1 / (evir[:, None] - eocc[None, :])

mo1_base = np.zeros_like(hsmo)
mo1_base[:, nocc:] = -hsmo[:, nocc:] * e_ai
mo1_base[:, :nocc] = -s1mo[:, :nocc] * 0.5

fvind = hessian.rhf.gen_vind(mf, mo_coeff, mo_occ)

def vind_vo(mo1):
    mo1 = mo1.reshape(-1, nmo, nocc)
    v = fvind(mo1).reshape(-1, nmo, nocc)
    v[:, nocc:, :] *= e_ai
    v[:, :nocc, :] = 0
    return v.reshape(-1, nmo * nocc)

mo1_base = mo1_base.reshape(-1, nmo * nocc)

## Other implementation of krylov

In [11]:
def krylov_glm(fx, b, x0=None, tol=1e-10):
    """Solve A*x = b using GMRES with Arnoldi iteration.

    Parameters
    ----------
    fx : callable
        Linear operator A, mapping [nset, n] -> [nset, n].
    b : ndarray, shape [nset, n]
        Right-hand sides (nset simultaneous systems).
    x0 : ndarray, shape [nset, n], optional
        Initial guess. Defaults to zero.
    tol : float
        Convergence tolerance (approximately ||A*x - b||_inf).
        Internally uses the L2 residual norm maintained by Givens
        rotations; since ||r||_inf <= ||r||_2, this guarantees
        ||A*x - b||_inf < tol.

    Returns
    -------
    x : ndarray, shape [nset, n]
        Approximate solution.

    Notes
    -----
    The least-squares problem min ||beta*e1 - H*y|| that arises at
    each GMRES step is handled by incrementally maintaining the QR
    factorization of the upper Hessenberg matrix H using Givens
    rotations.  This avoids calling lstsq or qr (the numpy/scipy
    functions) while being numerically stable — Givens rotations do
    not square the condition number the way the normal equations would.
    """
    nset, n = b.shape

    # --- initial residual ---------------------------------------------------
    if x0 is None:
        x0 = np.zeros_like(b)
        r0 = b.copy()                        # A*0 = 0 for linear operators
    else:
        r0 = b - fx(x0)                     # 1 fx call

    beta = np.linalg.norm(r0, axis=1)       # [nset] L2 norms

    if np.all(beta < tol):
        return x0.copy()

    # --- first Arnoldi vector -----------------------------------------------
    beta_safe = np.where(beta > 1e-300, beta, 1.0)
    V = [r0 / beta_safe[:, None]]
    V[0][beta < tol] = 0.0                  # zero out already-converged RHS

    max_iter = n

    # GMRES incremental QR via Givens rotations
    g = np.zeros((nset, max_iter + 1))      # transformed RHS
    g[:, 0] = beta

    cs = []          # Givens rotation parameters per step: (c, s) each [nset]
    R_cols = []      # columns of R (upper triangular factor)
    m = 0

    for k in range(max_iter):
        print(f"GMRES iteration {k}")
        # --- matrix-vector product (batched) -------------------------------
        w = fx(V[k])                        # 1 fx call per iteration

        # --- modified Gram-Schmidt with re-orthogonalization (2 passes) ---
        h = np.zeros((nset, k + 2))
        for _ in range(2):
            for j in range(k + 1):
                dot = np.sum(V[j] * w, axis=1)
                h[:, j] += dot
                w -= dot[:, None] * V[j]

        h[:, k + 1] = np.linalg.norm(w, axis=1)

        # next Arnoldi vector
        hnorm = np.where(h[:, k + 1] > 1e-15, h[:, k + 1], 1.0)
        v_next = w / hnorm[:, None]
        v_next[h[:, k + 1] <= 1e-15] = 0.0
        V.append(v_next)

        # --- apply previous Givens rotations to new Hessenberg column ------
        for i in range(k):
            c_i, s_i = cs[i]                # each shape [nset]
            temp = c_i * h[:, i] + s_i * h[:, i + 1]
            h[:, i + 1] = -s_i * h[:, i] + c_i * h[:, i + 1]
            h[:, i] = temp

        # --- new Givens rotation to zero out h[:, k+1] ---------------------
        r_rot = np.sqrt(h[:, k] ** 2 + h[:, k + 1] ** 2)
        c_k = np.where(r_rot > 1e-300, h[:, k] / r_rot, 1.0)
        s_k = np.where(r_rot > 1e-300, h[:, k + 1] / r_rot, 0.0)
        cs.append((c_k, s_k))

        # apply new rotation
        h[:, k] = c_k * h[:, k] + s_k * h[:, k + 1]
        # h[:, k+1] is now zero (not stored)

        # apply new rotation to g
        temp = c_k * g[:, k] + s_k * g[:, k + 1]
        g[:, k + 1] = -s_k * g[:, k] + c_k * g[:, k + 1]
        g[:, k] = temp

        # store R column (upper part only)
        R_cols.append(h[:, :k + 1].copy())

        m = k + 1

        # --- convergence check: |g[:, m]| = ||r_m||_2 ---------------------
        if np.all(np.abs(g[:, m]) < tol):
            break

    # --- solve R * y = g[:, :m] via back-substitution ----------------------
    # build R as [nset, m, m] upper triangular
    R = np.zeros((nset, m, m))
    for j in range(m):
        R[:, :j + 1, j] = R_cols[j]

    y_all = np.zeros((nset, m))
    for i in range(nset):
        y_all[i] = solve_triangular(R[i], g[i, :m], lower=False)

    # --- assemble solution: x = x0 + V * y --------------------------------
    x = x0.copy()
    for j in range(m):
        x += y_all[:, j:j + 1] * V[j]

    return x


In [18]:
def krylov_block(aop, b, x0=None, tol=1e-10, max_cycle=30, lindep=1e-13):
    """Block Krylov subspace method to solve (1+a)*x = b.

    At each cycle, the current trial block is multiplied by aop and
    orthogonalized against the full subspace.  This adds nroots new
    directions per cycle (one per surviving right-hand side), making
    it much faster than single-vector GMRES for multi-RHS problems.

    The basis vectors are kept non-normalized (orthogonal but not
    orthonormal), and their squared norms are tracked in `innerprod`.
    This causes trial vectors to naturally "deflate" as the solution
    converges, providing an accurate and inexpensive convergence signal
    without requiring the actual residual to be computed.

    Parameters
    ----------
    aop : callable
        Linear operator a, mapping (nblock, n) -> (nblock, n).
        The equation solved is (1 + aop) * x = b.
    b : ndarray, shape (nset, n) or (n,)
        Right-hand sides.
    x0 : ndarray, shape (nset, n) or (n,), optional
        Initial guess.
    tol : float
        Convergence tolerance.  Iteration stops when
        max(||new_trial_vec_i||^2) < max(lindep, tol^2).
    max_cycle : int
        Maximum number of block cycles.
    lindep : float
        Linear dependency threshold.  Vectors with ||v||^2 < lindep
        are dropped from the subspace.

    Returns
    -------
    x : ndarray, same shape as b
        Approximate solution of (1 + aop) * x = b.
    """
    if b.ndim == 1:
        b = b.reshape(1, -1)
        was_1d = True
    else:
        was_1d = False
    nset_l, ndim_l = b.shape

    if x0 is not None:
        if x0.ndim == 1:
            x0 = x0.reshape(1, -1)
        b = b - (x0 + aop(x0))

    # MGS that keeps non-normalized vectors and tracks ||v||^2
    def _orth_block(vec_list):
        result = []
        norms_sq = []
        for vi in vec_list:
            vi = vi.copy()
            for j in range(len(result)):
                coeff = np.dot(vi, result[j]) / norms_sq[j]
                vi -= coeff * result[j]
            nsq = np.dot(vi, vi).real
            if nsq > lindep:
                result.append(vi)
                norms_sq.append(nsq)
        return result, norms_sq

    # Initialize: orthogonalize b
    x1_list, innerprod = _orth_block([b[i] for i in range(nset_l)])

    if not x1_list:
        result = np.zeros((nset_l, ndim_l))
        if x0 is not None:
            result += x0
        return result[0] if was_1d else result

    xs = []              # basis vectors (orthogonal, non-normalized)
    ax_list = []         # aop(xs[i]) for each basis vector
    all_innerprod = []    # ||xs[i]||^2

    for cycle in range(max_cycle):
        # Apply operator to current trial block
        x1_arr = np.array(x1_list)
        axt = aop(x1_arr)
        if axt.ndim == 1:
            axt = axt.reshape(1, -1)

        # Store basis vectors and aop results
        for i in range(len(x1_list)):
            xs.append(x1_list[i].copy())
            ax_list.append(axt[i].copy())
            all_innerprod.append(innerprod[i])

        # Orthogonalize axt against full subspace
        # For non-normalized orthogonal xs[i] with ||xs[i]||^2 = all_innerprod[i]:
        #   proj_coeff = dot(v, xs[i]) / all_innerprod[i]
        x1_new = axt.copy()
        for i in range(len(xs)):
            xsi = xs[i]
            w = x1_new @ xsi / all_innerprod[i]    # (nblock,) coefficients
            x1_new -= np.outer(w, xsi)

        # Orthogonalize among themselves via MGS
        x1_list, innerprod = _orth_block(x1_new)

        max_innerprod = max(innerprod) if innerprod else 0
        r = np.sqrt(max_innerprod)

        print(f"Cycle {cycle}: max ||new_trial_vec_i||^2 = {max_innerprod:.3e}, max ||new_trial_vec_i|| = {r:.3e}")

        # Convergence check (same as PySCF)
        if max_innerprod < max(lindep, tol**2):
            break

    # Build and solve small system: (I + A_projected) * c = b_projected
    nd = len(xs)
    Xs = np.array(xs)
    AX = np.array(ax_list)

    # h[i,j] = dot(xs[i], aop(xs[j])) + delta_ij * ||xs[i]||^2
    h = Xs @ AX.T
    for i in range(nd):
        h[i, i] += all_innerprod[i]

    # g[i,k] = dot(b[k], xs[i])
    g = Xs @ b.T

    c = np.linalg.solve(h, g)
    x = c.T @ Xs

    if x0 is not None:
        x += x0

    return x[0] if was_1d else x

## Compare different solvers

In [13]:
mo1_pyscf = lib.krylov(vind_vo, mo1_base, tol=1e-8, verbose=5)

krylov cycle 0  r = 0.0688479
krylov cycle 1  r = 0.0149531
krylov cycle 2  r = 0.00172213
krylov cycle 3  r = 0.000175467
krylov cycle 4  r = 1.34441e-05
krylov cycle 5  r = 1.02565e-06
krylov cycle 6  r = 0


In [14]:
def vind_vo_plus(x):
    return vind_vo(x) + x

In [15]:
mo1_glm = krylov_glm(vind_vo_plus, mo1_base, tol=1e-8)
assert np.allclose(mo1_glm, mo1_pyscf, atol=1e-6, rtol=1e-4)

GMRES iteration 0
GMRES iteration 1
GMRES iteration 2
GMRES iteration 3
GMRES iteration 4
GMRES iteration 5
GMRES iteration 6
GMRES iteration 7
GMRES iteration 8
GMRES iteration 9
GMRES iteration 10
GMRES iteration 11
GMRES iteration 12


In [19]:
mo1_block = krylov_block(vind_vo, mo1_base, tol=1e-8)

Cycle 0: max ||new_trial_vec_i||^2 = 4.740e-03, max ||new_trial_vec_i|| = 6.885e-02
Cycle 1: max ||new_trial_vec_i||^2 = 2.236e-04, max ||new_trial_vec_i|| = 1.495e-02
Cycle 2: max ||new_trial_vec_i||^2 = 2.966e-06, max ||new_trial_vec_i|| = 1.722e-03
Cycle 3: max ||new_trial_vec_i||^2 = 3.079e-08, max ||new_trial_vec_i|| = 1.755e-04
Cycle 4: max ||new_trial_vec_i||^2 = 1.807e-10, max ||new_trial_vec_i|| = 1.344e-05
Cycle 5: max ||new_trial_vec_i||^2 = 1.052e-12, max ||new_trial_vec_i|| = 1.026e-06
Cycle 6: max ||new_trial_vec_i||^2 = 0.000e+00, max ||new_trial_vec_i|| = 0.000e+00
